## Setup
Helper functions for reading from Bronze and writing to Silver, so each cleaning step below is two lines instead of a repeated path string.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "transfermarkt"

def bronze(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/bronze/{t}")

def to_silver(df, name):
    df.write.mode("overwrite").parquet(f"/Volumes/{CATALOG}/{SCHEMA}/silver/{name}")

## Tables that only need deduplication
`competitions`, `clubs`, `countries`, `national_teams`, `club_games`, `game_events`, `game_lineups`, and `transfers` have no known type or date issue at this stage — the only cleaning they need is dropping exact duplicate rows.

In [0]:
passthrough = ["competitions", "clubs", "countries", "national_teams",
               "club_games", "game_events", "game_lineups", "transfers"]

for t in passthrough:
    to_silver(bronze(t).dropDuplicates(), t)

## `players`
Deduped on the real natural key, `player_id` — not a full-row match, since two rows could differ in a free-text field like `agent_name` and still be the same player. `date_of_birth` is cast from `timestamp` down to `date`; this can't introduce new nulls, since it was already a proper temporal type in Bronze, not a string being parsed for the first time.

In [0]:
players = (bronze("players")
           .dropDuplicates(["player_id"])
           .withColumn("date_of_birth", F.to_date("date_of_birth")))
to_silver(players, "players")
print("players: date_of_birth nulls after cast:",
      players.filter(F.col("date_of_birth").isNull()).count())

## `games`
Deduped on `game_id`. `date` was already typed correctly in Bronze, so this cast is a formality — the null count confirms that.

In [0]:
games = (bronze("games")
         .dropDuplicates(["game_id"])
         .withColumn("date", F.to_date("date")))
to_silver(games, "games")
print("games: date nulls after cast:", games.filter(F.col("date").isNull()).count())

## `player_valuations` and `appearances`
Both already have a correctly-typed `date` in Bronze, same reasoning as `games`. No deduplication here — unlike `players` or `games`, we haven't yet confirmed `appearance_id` is actually a unique key rather than just assumed it; that check happens in the EDA notebook, not here.

In [0]:
player_valuations = bronze("player_valuations").withColumn("date", F.to_date("date"))
to_silver(player_valuations, "player_valuations")

appearances = bronze("appearances").withColumn("date", F.to_date("date"))
to_silver(appearances, "appearances")